# Kaggle launcher

A cell-by-cell launcher that packages a TAAF benchmark for Kaggle and
(optionally) uploads it. The whole point of the framework's deploy story is
that this is just **a benchmark + a solver + a Kaggle target**, so that's all
this notebook does — built directly, no CLI or per-solver helper module.

You run this **launcher** locally (or on a Tufa submit host); it does not run on
Kaggle itself — it *generates* the notebook that runs up there (see
[`taaf.deploy_kaggle`](../src/taaf/deploy_kaggle.py)).

It uses `SolverRandom` as the example solver; the *Build the solver* cell marks
where you'd drop your own.

Spec: R2.24 / R2.51–R2.61 in [`docs/requirements.md`](../docs/requirements.md)
— status: [`docs/requirement_status.md`](../docs/requirement_status.md).

## Two ways to run this

- **Dry run (default, `DRY_RUN = True`)** — writes the Kaggle bundles under
  `JOB_DIR/kaggle/` and touches nothing on Kaggle. No credentials needed. Safe
  "does packaging work?" smoke test; run-all is harmless.
- **Real upload (`DRY_RUN = False`)** — creates/updates your per-user Kaggle
  notebook + source dataset via the Kaggle CLI, **and immediately runs the
  notebook on Kaggle** (see the note in the Deploy section). Flip the flag below
  when ready.

### Credentials (only needed for a real upload)

Resolved by `taaf.deploy_kaggle`, in order: `KAGGLE_USERNAME` + `KAGGLE_KEY`
env vars → a `.env` found by walking up from the cwd → `~/.kaggle/kaggle.json`.
You also need the Kaggle CLI on PATH (`pip install kaggle`).

In [ ]:
from datetime import datetime
from pathlib import Path

import taaf

# Run-environment knobs. Everything else is set inline below, right next to
# the benchmark / solver / target it belongs to.
DRY_RUN = True   # True: package locally only. False: create/update + run the real Kaggle notebook.
RUN_AS_SUBMISSION = False   # True: emulate submission mode (no logging/diagnostics). A real
                           # competition rerun forces this on regardless of this flag.
JOB_DIR = Path("/tmp") / f"taaf-kaggle-smoke-{datetime.now():%Y%m%d-%H%M%S}"

print("DRY_RUN          :", DRY_RUN)
print("RUN_AS_SUBMISSION:", RUN_AS_SUBMISSION)
print("JOB_DIR          :", JOB_DIR)

## 1. Build the solver

Swap the `SolverRandom(...)` line for your own `taaf.solver.Solver` subclass —
everything downstream (benchmark, target, deploy) is unchanged. See *Deploying
your own solver* near the bottom for the three things to get right.

In [ ]:
solver = taaf.solver_examples.SolverRandom(   # ← your taaf.solver.Solver subclass goes here
    seed=0,
    max_actions_per_game=1000,   # ← lower for a quicker run; None to play each game out fully
    click_on_color=True,  # better performance on click games
    stop_on_error=True # set to True to have an error on one game crash the whole run (robustness test)
)

print("solver:", solver.label)

## 2. Build the benchmark

A benchmark is just games + solver + passes. `make_benchmark_kaggle_official_110`
is the standard submission-shaped benchmark (R2.57): the 25 official games
repeated to 110 `GameAPI` entries, `n_passes=1`.

This notebook passes `competition_sim=True`, so the run targets a **local
competition-arcade simulator** (R11.13) started on the worker — the
submission-shaped arcade (one shared scorecard, hidden baselines, cloned game
IDs) — rather than the plain offline bundle. That makes the smoke test exercise
the same arcade path a real submission hits. Pass `competition_sim=False` for
offline play, or swap in your own game list for something smaller/faster.

In [ ]:
benchmark = taaf.standard_benchmarks.make_benchmark_kaggle_official_110(solver=solver, competition_sim=True) 
benchmark.label = "taaf-kaggle-smoke"   # ← run label (shows up in diagnostics)
benchmark.job_dir = JOB_DIR

# ← competition_sim=True targets a local competition-arcade simulator (R11.13) started on
#   the worker — the submission-shaped arcade. Pass competition_sim=False for the plain
#   offline re-arc bundle. Want different / fewer games? build the Benchmark directly:
# benchmark = taaf.benchmark.Benchmark(
#     label="taaf-kaggle-smoke",
#     games=[taaf.game_api.GameAPI(env_name=g) for g in ("ls20", "ft09")],  # ← your games
#     solver=solver,
#     n_passes=1,   # competition arcade requires a single pass
#     job_dir=JOB_DIR,
# )

print("games:", len(benchmark.games), "| passes:", benchmark.n_passes)

## 3. Build the target

The Kaggle target carries the *where/how it runs on Kaggle* config. Most fields
have sensible defaults (R2.52: private, internet off, RTX6000 GPU); the
commented lines below are the ones you're most likely to touch.

Set `make_share_version=True` to build the **public share** variant (R2.61) instead: a minimal, Kaggle-only notebook (`taaf_kaggle_run_share.ipynb`) that always plays the live competition Arcade, leaves `re-arc-3` out of the source dataset, and `-share`-suffixes the default kernel/dataset slugs (explicit slugs are untouched).

In [ ]:
target = taaf.deploy_kaggle.KaggleTarget(
    kernel_slug="taaf-kaggle-smoke",   # ← your per-user Kaggle notebook slug
    dry_run=DRY_RUN,
    cpu_only=True,            # ← False for the RTX6000 default (real LLM solvers need a GPU)
    max_runtime_s=9 * 3600,   # ← Kaggle notebook runtime budget (soft deadline = this - 10 min)
    # username=None,                 # ← None resolves from KAGGLE_USERNAME / ~/.kaggle
    # dataset_ref=None,              # ← None -> "<username>/taaf-release-source"
    run_as_submission=RUN_AS_SUBMISSION,  # ← a real competition rerun forces this on anyway
    # make_share_version=True,       # ← public share notebook (R2.61: minimal, Kaggle-only; drops re-arc-3; -share slugs)
    # extra_source_repos=[Path("/path/to/my-solver")],  # ← bundle a non-editable solver repo
)

print("dry_run:", target.dry_run, "| cpu_only:", target.cpu_only,
      "| accelerator:", None if target.cpu_only else target.accelerator)

## 4. Deploy

`await benchmark.deploy(target)` stages the bundles under `JOB_DIR/kaggle/` and
— unless `DRY_RUN` — creates/versions the source dataset and pushes the notebook
through the Kaggle CLI. Notebook cells run in an event loop, so top-level
`await` works directly.

**A real upload auto-runs the notebook:** `kaggle kernels push` commits *and*
executes the kernel — there is no push-without-run in the Kaggle API. So with
`DRY_RUN = False`, the notebook starts running on Kaggle (and bills compute) as
soon as this cell finishes. If you didn't mean to run it, cancel it yourself
from the Kaggle UI (the notebook's *Versions* / *Logs* → stop the run).

In [ ]:
handle = await benchmark.deploy(target)

print("kernel_id   :", handle.kernel_id)
print("dataset_ref :", handle.dataset_ref)
print("uploaded    :", handle.uploaded)
print("job_dir     :", handle.job_dir)
if handle.uploaded:
    print("notebook    : https://www.kaggle.com/code/" + handle.kernel_id)

## 5. Inspect what was staged

Both dry-run and real deploys write the same local bundle under
`JOB_DIR/kaggle/`: a `source-dataset/` (pickled benchmark + target, generated
runner, source snapshots) and a `kernel/` (the generated notebook +
`kernel-metadata.json`).

In [ ]:
staging = handle.job_dir / "kaggle"
for p in sorted(staging.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(staging)}  ({p.stat().st_size} bytes)")

In [ ]:
# Eyeball the kernel config the upload would use (GPU / internet / attached datasets).
print((staging / "kernel" / "kernel-metadata.json").read_text())

## 6. (Real upload only) wait for the run

When `uploaded`, the kernel runs asynchronously on Kaggle. `handle.wait()` polls
`kaggle kernels status` and then pulls `benchmark.json` back from the output:

```python
result = handle.wait()
print(result.game_runs[0].state, result.game_runs[0].final_score)
```

Submission-mode notebooks suppress writes, so there's no `benchmark.json` —
`wait()` raises after the kernel finishes (expected). You can also reattach from
a fresh session via `taaf.deploy.DeploymentHandle.attach(JOB_DIR)`.

## Deploying your own solver

Replace the solver in cell 1 with your own `Solver` subclass; the rest is
unchanged. Three things to get right:

1. **Picklable until `setup()`** (R12.02). The solver is pickled into the bundle
   and unpickled on Kaggle, so build LLM engines / open files in `_setup()`,
   not `__init__`.
2. **Your code must be in the source snapshot.** `deploy` bundles every
   *editable* install in the launcher's venv (PEP 610). If your solver lives in
   its own repo, `pip install -e` it here, or pass `extra_source_repos=[...]`.
3. **Declare solver-specific Kaggle needs on the solver** — `deploy` picks them
   up automatically (R2.54–R2.56): `kaggle_dataset_sources` (model weights),
   `kaggle_pip_install_packages` / `kaggle_setup_commands` (install vllm, set
   env before load), `kaggle_teardown_commands` (cleanup). Submission mode
   reaches the solver via `self.runtime_environment.actual_run_as_submission` —
   read that rather than sniffing env vars.

## Where things land

| Path under `JOB_DIR/kaggle/`                                    | Written by                          | Purpose                                            |
|-----------------------------------------------------------------|-------------------------------------|----------------------------------------------------|
| `source-dataset/benchmark_initial.pkl`                          | `KaggleTarget.deploy`               | Pickled benchmark the Kaggle runner loads          |
| `source-dataset/deploy_target.pkl`                              | `KaggleTarget.deploy`               | Pickled target → `runtime_environment` in `bm.run` |
| `source-dataset/src/<repo>/...`                                 | `snapshot_editable_sources` (R2.35) | Source bundle for the run                          |
| `source-dataset/setup_commands.json` / `teardown_commands.json` | `KaggleTarget.deploy`               | Solver/target shell hooks (R2.56)                  |
| `source-dataset/preamble.txt`                                   | `KaggleTarget.deploy`               | Label / solver repr / git status (R2.31.2)         |
| `kernel/<notebook>.ipynb`                                       | `KaggleTarget.deploy`               | Generated notebook that runs on Kaggle             |
| `kernel/kernel-metadata.json`                                   | `KaggleTarget.deploy`               | GPU / internet / dataset-attachment config         |
| `deploy_meta.json` (in `JOB_DIR`)                               | `write_deploy_meta` (R2.39)         | What `DeploymentHandle.attach(JOB_DIR)` reads      |

### Source

- [`src/taaf/deploy_kaggle.py`](../src/taaf/deploy_kaggle.py) — `KaggleTarget` / `KaggleHandle`
- [`src/taaf/standard_benchmarks.py`](../src/taaf/standard_benchmarks.py) — `make_benchmark_kaggle_official_110` (R2.57)
- [`src/taaf/competition_arcade.py`](../src/taaf/competition_arcade.py) — local competition-arcade simulator (R11.13)
- [`docs/requirements.md`](../docs/requirements.md) — R2.24 / R2.51–R2.61 spec